In [1]:
import os
import json
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn,optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models, ops
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.rpn import RegionProposalNetwork, RPNHead
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.faster_rcnn import TwoMLPHead, FastRCNNPredictor
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from scipy.io import loadmat
from scipy.spatial.transform import Rotation as R
import numpy as np
import tqdm

def get_discrete_classes(m):
    """
    Computes m uniformly distributed random rotations parametrized as unit quaternions.
    
    Args:
        m (int): Number of random rotations to generate
        
    Returns:
        numpy.ndarray: Matrix of shape (m, 4) containing unit quaternions
    """
    x0 = np.random.rand(m)
    x1 = np.random.rand(m)
    x2 = np.random.rand(m)
    
    
    theta1 = 2 * np.pi * x1
    theta2 = 2 * np.pi * x2
    
    
    s1 = np.sin(theta1)
    s2 = np.sin(theta2)
    c1 = np.cos(theta1)
    c2 = np.cos(theta2)
    
    
    r1 = np.sqrt(1 - x0)
    r2 = np.sqrt(x0)
    
    
    quats = np.column_stack([s1 * r1, c1 * r1, s2 * r2, c2 * r2])
    
    return quats

def _get_quat_bins(qPose, qClass, numNeighbors):
    ''' For each quaternion in `qPose`, find the nearest quaternion in `qClass` and
        the weights associated with distance to each class. Based on the original MATLAB 
        implementation by Sumant Sharma.
    Arguments:
        qPose:  (4,)  array of unit quaternion (scalar-first)
        qClass: (N,4) matrix of unit quaternion classes (scalar-first)
        numNeighbors: (int) number of `qClass` covering each of `qOise`
    Returns:
        nClasses: (numNeighbors,) array of closest quaternion classes
        nWeights: (numNeighbors,) array of weights of each quaternion classes
    '''
    # Quaternion classes into scalar-last for scipy
    q      = R.from_quat(qPose[[1,2,3,0]])
    qClass = R.from_quat(qClass[:,[1,2,3,0]])

    # Orientation diff. w.r.t. each entries in qClass [numClasses, 4]
    qDiff = q.inv() * qClass
    qDiff = qDiff.as_quat()

    # Angular distance w.r.t. each entries of qClass [rad]
    angleVec = 2 * np.arccos(np.abs(qDiff[:,-1])) # scalar-last

    # nAngles:  Smallest angular distances [rad]
    # nCLasses: Class indices of such entries
    sortIdx  = np.argsort(angleVec)
    nClasses = sortIdx[:numNeighbors]
    nAngles  = angleVec[nClasses]

    # Angular distances into weights
    nWeights = 1.0 - nAngles / np.pi**2 # (numNeighbors,)
    nWeights = nWeights / np.sum(nWeights)

    return nClasses, nWeights

def softmax_cross_entropy_with_logits(logits, target, reduction='mean'):
    """ Implementation of tensorflow's function of the same name
        - logits [B x C]
        - target [B x C]
    """
    loss = -torch.sum(target.detach() * F.log_softmax(logits, dim=1), dim=1) # [B,]
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss
    
def load_tango_3d_keypoints(mat_dir):
    vertices  = loadmat(mat_dir)['tango3Dpoints'] # [3 x 11]
    corners3D = np.transpose(np.array(vertices, dtype=np.float32)) # [11 x 3]

    return corners3D

def load_camera_intrinsics(camera_json):
    with open(camera_json) as f:
        cam = json.load(f)
    cameraMatrix = np.array(cam['cameraMatrix'], dtype=np.float32)
    distCoeffs   = np.array(cam['distCoeffs'],   dtype=np.float32)

    return cameraMatrix, distCoeffs

def quat2dcm(q):
    """ Computing direction cosine matrix from quaternion, adapted from PyNav. 
    Arguments:
        q: (4,) numpy.ndarray - unit quaternion (scalar-first)
    Returns:
        dcm: (3,3) numpy.ndarray - corresponding DCM
    """

    # normalizing quaternion
    q = q/np.linalg.norm(q)

    q0 = q[0]
    q1 = q[1]
    q2 = q[2]
    q3 = q[3]

    dcm = np.zeros((3, 3))

    dcm[0, 0] = 2 * q0 ** 2 - 1 + 2 * q1 ** 2
    dcm[1, 1] = 2 * q0 ** 2 - 1 + 2 * q2 ** 2
    dcm[2, 2] = 2 * q0 ** 2 - 1 + 2 * q3 ** 2

    dcm[0, 1] = 2 * q1 * q2 + 2 * q0 * q3
    dcm[0, 2] = 2 * q1 * q3 - 2 * q0 * q2

    dcm[1, 0] = 2 * q1 * q2 - 2 * q0 * q3
    dcm[1, 2] = 2 * q2 * q3 + 2 * q0 * q1

    dcm[2, 0] = 2 * q1 * q3 + 2 * q0 * q2
    dcm[2, 1] = 2 * q2 * q3 - 2 * q0 * q1

    return dcm

def project_keypoints(q_vbs2tango, r_Vo2To_vbs, cameraMatrix, distCoeffs, keypoints):
    ''' Project keypoints.
    Arguments:
        q_vbs2tango:  (4,) numpy.ndarray - unit quaternion from VBS to Tango frame
        r_Vo2To_vbs:  (3,) numpy.ndarray - position vector from VBS to Tango in VBS frame (m)
        cameraMatrix: (3,3) numpy.ndarray - camera intrinsics matrix
        distCoeffs:   (5,) numpy.ndarray - camera distortion coefficients in OpenCV convention
        keypoints:    (3,N) or (N,3) numpy.ndarray - 3D keypoint locations (m)
    Returns:
        points2D: (2,N) numpy.ndarray - projected points (pix)
    '''
    # Size check (3,N)
    if keypoints.shape[0] != 3:
        keypoints = np.transpose(keypoints)

    # Keypoints into 4 x N homogenous coordinates
    keypoints = np.vstack((keypoints, np.ones((1, keypoints.shape[1]))))

    # transformation to image frame
    pose_mat = np.hstack((np.transpose(quat2dcm(q_vbs2tango)),
                          np.expand_dims(r_Vo2To_vbs, 1)))
    xyz      = np.dot(pose_mat, keypoints) # [3 x N]
    x0, y0   = xyz[0,:] / xyz[2,:], xyz[1,:] / xyz[2,:] # [1 x N] each

    # apply distortion
    r2 = x0*x0 + y0*y0
    cdist = 1 + distCoeffs[0]*r2 + distCoeffs[1]*r2*r2 + distCoeffs[4]*r2*r2*r2
    x  = x0*cdist + distCoeffs[2]*2*x0*y0 + distCoeffs[3]*(r2 + 2*x0*x0)
    y  = y0*cdist + distCoeffs[2]*(r2 + 2*y0*y0) + distCoeffs[3]*2*x0*y0

    # apply camera matrix
    points2D = np.vstack((cameraMatrix[0,0]*x + cameraMatrix[0,2],
                          cameraMatrix[1,1]*y + cameraMatrix[1,2]))

    return points2D

class AverageMeter(object):
    """ Computes and stores the average and current value.
    Can handle both scalar values and numpy arrays.
    """
    def __init__(self, unit='-', is_vector=False):
        self.reset()
        self.unit = unit
        self.is_vector = is_vector

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        
    def update(self, val, n=1):
        self.val = val
        
        # Handle initialization for vectors
        if self.count == 0 and isinstance(val, np.ndarray):
            self.sum = np.zeros_like(val, dtype=np.float64)
            
        # Update sum and compute average
        if isinstance(val, np.ndarray):
            self.sum = self.sum + val * n
        else:
            self.sum += val * n
            
        self.count += n
        
        if self.count > 0:
            if isinstance(self.sum, np.ndarray):
                self.avg = self.sum / self.count
            else:
                self.avg = self.sum / self.count
                
    def magnitude(self):
        """Return magnitude for vector quantities"""
        if isinstance(self.avg, np.ndarray):
            return np.linalg.norm(self.avg)
        return self.avg

def weighted_mean_quaternion(qs, weights=None):
    ''' Compute weighted mean of N unit quaternions.
    Arguments:
        qs: (N, 4) or (4, N) numpy.ndarray - unit quaternions (scalar-first)
    Returns:
        q: (4,) numpy.ndarray - weighted mean unit quaternion (scalar-first)
    '''
    # Size check
    if qs.shape[1] != 4:
        qs = np.transpose(qs) # (N,4)

    # Scipy uses scalar-last convention
    qs = qs[:,[1,2,3,0]]

    # Weights?
    if weights is None:
        weights = np.ones((qs.shape[0],), dtype=np.float32)

    # Quaternions to rotation matrices
    Rs = R.from_quat(qs)

    # Weighted average
    q = Rs.mean(weights).as_quat() # (4,)

    # Back to scalar-first convention
    q = q[[3,0,1,2]]
    return q

def compute_position_spn(q_vbs2tango, bbox, corners3D, cameraMatrix, distCoeffs=np.zeros((1,5))):
    ''' Compute position vector for SPN model
    Arguments:
        q_vbs2tango: (4,) numpy.ndarray - predicted unit quaternion (scalar-first)
        bbox:        (4,) numpy.ndarray - bounding box [xmin, xmax, ymin, ymax] (pix)
        ...
    Returns:
        r_Vo2To_vbs: (3,) numpy.ndarray - predicted position vector (m)
    '''
    maxModelLength = 1.246 # [m] for Tango

    # Bounding box decomposition
    xmin, ymin, width, height = bbox[0], bbox[2], bbox[1]-bbox[0], bbox[3]-bbox[2]

    # Initial position guess based on similar triangles
    boxSize   = np.sqrt(width**2 + height**2)
    boxCenter = np.array([xmin + width/2.0, ymin + height/2.0])
    offsetPx  = np.array([boxCenter[0] - cameraMatrix[0,2],
                          boxCenter[1] - cameraMatrix[1,2]])
    az = np.arctan(offsetPx[0]/cameraMatrix[0,0]) # [rad]
    el = np.arctan(offsetPx[1]/cameraMatrix[1,1])
    range_wge = cameraMatrix[0,0] * maxModelLength / boxSize # [m]
    Ry = R.from_euler('y', -az).as_matrix()
    Rx = R.from_euler('x', -el).as_matrix()
    r_Vo2To_vbs = Ry @ Rx @ np.reshape(np.array([0, 0, range_wge]), (3,1))

    # NEWTON's METHOD
    maxIter = 50
    tolerance = 5e-10
    iter = 0
    dx = 1 + 1e-15

    # Initialize betas
    beta_old = np.squeeze(r_Vo2To_vbs)

    while dx > tolerance and iter <= maxIter:
        # Compute extreme reprojected points in VBS frame
        r_Vo2X_vbs = _compute_extremal_points(q_vbs2tango, beta_old, corners3D, cameraMatrix) # [4 x 3]

        # Compute update to beta
        r = _calc_residuals(r_Vo2X_vbs, cameraMatrix, distCoeffs, beta_old, bbox)
        J = _calc_jacobian(r_Vo2X_vbs, cameraMatrix, distCoeffs, beta_old)
        beta_new = beta_old - np.squeeze(np.linalg.inv(np.transpose(J) @ J) @ np.transpose(J) @ np.reshape(r, (4,1)))

        # Compute change between new and oldbeta
        dx = np.linalg.norm(beta_new - beta_old)

        # Updates
        iter = iter + 1
        beta_old = beta_new

    r_Vo2To_vbs = beta_new

    return r_Vo2To_vbs

def _compute_extremal_points(q_vbs2tango, r_Vo2To_vbs, tangoPoints, cameraMatrix):
    ''' Compute the extremal points of the Tango model given orientation and position estimates '''
    reprImagePoints = project_keypoints(q_vbs2tango, r_Vo2To_vbs,
                                cameraMatrix, np.zeros((5,)), tangoPoints)
    idx1 = np.argmin(reprImagePoints[0]) # xmin
    idx2 = np.argmin(reprImagePoints[1]) # ymin
    idx3 = np.argmax(reprImagePoints[0]) # xmax
    idx4 = np.argmax(reprImagePoints[1]) # ymax

    if tangoPoints.shape[0] != 3:
        tangoPoints = np.transpose(tangoPoints)
    tangoPoints_vbs = np.transpose(quat2dcm(q_vbs2tango)) @ tangoPoints

    r_Vo2X_vbs = np.zeros((4, 3))
    r_Vo2X_vbs[0] = tangoPoints_vbs[:,idx1] # left-most point
    r_Vo2X_vbs[1] = tangoPoints_vbs[:,idx3] # right-most point
    r_Vo2X_vbs[2] = tangoPoints_vbs[:,idx2] # top-most point
    r_Vo2X_vbs[3] = tangoPoints_vbs[:,idx4] # bottom-most point

    return r_Vo2X_vbs

def _calc_residuals(r_Vo2X_vbs, cameraMatrix, distCoeffs, r_Vo2To_vbs, bbox):
    ''' Compute residuals of projected extremal points against the bounding box '''
    Tx, Ty, Tz = r_Vo2To_vbs
    Bx1, Bx2, By1, By2 = bbox

    xs, ys = [], []
    for ii in range(4):
        # Project
        Rx, Ry, Rz = r_Vo2X_vbs[ii]
        x0 = (Rx + Tx) / (Rz + Tz)
        y0 = (Ry + Ty) / (Rz + Tz)

        # Distortion
        r2 = x0*x0 + y0*y0
        cdist = 1 + distCoeffs[0]*r2 + distCoeffs[1]*r2*r2 + distCoeffs[4]*r2*r2*r2
        x  = x0*cdist + distCoeffs[2]*2*x0*y0 + distCoeffs[3]*(r2 + 2*x0*x0)
        y  = y0*cdist + distCoeffs[2]*(r2 + 2*y0*y0) + distCoeffs[3]*2*x0*y0

        # Apply camera
        xs.append(cameraMatrix[0,0]*x + cameraMatrix[0,2])
        ys.append(cameraMatrix[1,1]*y + cameraMatrix[1,2])

    # Residuals
    r1 = xs[0] - Bx1
    r2 = xs[1] - Bx2
    r3 = ys[2] - By1
    r4 = ys[3] - By2

    return np.array([r1, r2, r3, r4])

def _calc_jacobian(r_Vo2X_vbs, cameraMatrix, distCoeffs, r_Vo2To_vbs):
    ''' Compute jacobian of the residuals.
        Camera distortion coefficients are neglected at the moment.
    '''
    fx, fy = cameraMatrix[0,0], cameraMatrix[1,1]
    Tx, Ty, Tz = r_Vo2To_vbs
    Rx_left, Rz_left = r_Vo2X_vbs[0,0], r_Vo2X_vbs[0,2]
    Rx_right, Rz_right = r_Vo2X_vbs[1,0], r_Vo2X_vbs[1,2]
    Ry_top, Rz_top = r_Vo2X_vbs[2,1], r_Vo2X_vbs[2,2]
    Ry_bot, Rz_bot = r_Vo2X_vbs[3,1], r_Vo2X_vbs[3,2]

    # Left-most image feature
    dr1db1 = fx / (Rz_left + Tz)
    dr1db2 = 0
    dr1db3 = -fx * (Rx_left + Tx) / (Rz_left + Tz)**2

    # Right-most iamge feature
    dr2db1 = fx / (Rz_right + Tz)
    dr2db2 = 0
    dr2db3 = -fx * (Rx_right + Tx) / (Rz_right + Tz)**2

    # Top-most image feature
    dr3db1 = 0
    dr3db2 = fy / (Rz_top + Tz)
    dr3db3 = -fy * (Ry_top + Ty) / (Rz_top + Tz)**2

    # Bottom-most image feature
    dr4db1 = 0
    dr4db2 = fy / (Rz_bot + Tz)
    dr4db3 = -fy * (Ry_bot + Ty) / (Rz_bot + Tz)**2

    # Jacobian
    J = np.array([[dr1db1, dr1db2, dr1db3],
                  [dr2db1, dr2db2, dr2db3],
                  [dr3db1, dr3db2, dr3db3],
                  [dr4db1, dr4db2, dr4db3]], dtype=np.float32)

    return J

def error_translation(t_pr, t_gt):
    t_pr = np.reshape(t_pr, (3,))
    t_gt = np.reshape(t_gt, (3,))

    return t_gt - t_pr

def error_orientation(q_pr, q_gt):
    # q must be [qvec, qcos]
    q_pr = np.reshape(q_pr, (4,))
    q_gt = np.reshape(q_gt, (4,))

    qdot = np.abs(np.dot(q_pr, q_gt))
    qdot = np.minimum(qdot, 1.0)
    return np.rad2deg(2*np.arccos(qdot)) # [deg]

def calculate_iou(box1, box2):
    """
    Calculate the Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box1: [xmin, xmax, ymin, ymax] format (ground truth)
        box2: [xmin, ymin, xmax, ymax] format (detection format)
    
    Returns:
        IoU value
    """
    # Convert box2 from [xmin, ymin, xmax, ymax] to [xmin, xmax, ymin, ymax]
    box2_converted = [box2[0], box2[2], box2[1], box2[3]]
    
    # Calculate intersection area
    x_min_inter = max(box1[0], box2_converted[0])
    y_min_inter = max(box1[2], box2_converted[2])
    x_max_inter = min(box1[1], box2_converted[1])
    y_max_inter = min(box1[3], box2_converted[3])
    
    if x_max_inter < x_min_inter or y_max_inter < y_min_inter:
        return 0.0  # No intersection
    
    intersection = (x_max_inter - x_min_inter) * (y_max_inter - y_min_inter)
    
    # Calculate union area
    box1_area = (box1[1] - box1[0]) * (box1[3] - box1[2])
    box2_area = (box2_converted[1] - box2_converted[0]) * (box2_converted[3] - box2_converted[2])
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0.0

class Speed(Dataset):
    def __init__(self, images_dir, json_dir, transform=None):
        self.images_dir = images_dir
        self.json_dir = json_dir
        self.transform = transform

        self.num_classes = 5000
        self.num_neighbors = 25

        self.imagesList = []
        self.bboxList = []
        self.q_gtList = []
        self.t_gtList = []

        cnt=0
        
        self.attClassesMAT =  loadmat('/kaggle/input/mat-file/attitudeClasses.mat')['qClass']
        # self.attClassesMAT =  get_discrete_classes(1000)
        self.keypts3d = load_tango_3d_keypoints('/kaggle/input/mat-file/tangoPoints.mat') # https://www.desmos.com/3d/jud6lng9gn
        self.cameraMatrix, self.distCoeffs = load_camera_intrinsics('/kaggle/input/mat-file/camera.json')

        with open(self.json_dir, 'r') as f:
            annotations = json.load(f)
            lookup = { item['filename']: item for item in annotations }
            for filename in tqdm.tqdm(os.listdir(self.images_dir)):
                if filename not in lookup:
                    continue
                self.imagesList.append(os.path.join(self.images_dir, filename))

                q_vbs2tango = np.array(lookup[filename]["q_vbs2tango"], dtype=np.float32)
                r_Vo2To_vbs = np.array(lookup[filename]['r_Vo2To_vbs_true'], dtype=np.float32)

                self.q_gtList.append(q_vbs2tango)
                self.t_gtList.append(r_Vo2To_vbs)

                keypts2d = project_keypoints(q_vbs2tango, r_Vo2To_vbs, self.cameraMatrix, self.distCoeffs, self.keypts3d) # (2, 11)
                xmin = np.min(keypts2d[0])
                xmax = np.max(keypts2d[0])
                ymin = np.min(keypts2d[1])
                ymax = np.max(keypts2d[1])
                
                
                width = xmax - xmin
                height = ymax - ymin
                margin_x = width * 0.05
                margin_y = height * 0.05

                xmin = max(0, xmin - margin_x)
                xmax = xmax + margin_x
                ymin = max(0, ymin - margin_y)
                ymax = ymax + margin_y
                
                bbox = [xmin, xmax, ymin, ymax]

                self.bboxList.append(torch.tensor(bbox, dtype=torch.float32))

                cnt=cnt+1
                # if cnt>50:
                #     break

    def __len__(self):
        return len(self.imagesList)

    def __getitem__(self, idx):
        image_path = self.imagesList[idx]
        image = Image.open(image_path).convert('RGB')
        orig_w, orig_h = image.size
        q_gt = torch.from_numpy(self.q_gtList[idx])
        t_gt = (self.t_gtList[idx])
        bbox = self.bboxList[idx]

        if self.transform:
            image = self.transform(image)
            new_h, new_w = image.shape[1], image.shape[2]
            scale_x = new_w / orig_w
            scale_y = new_h / orig_h

            xmin, xmax, ymin, ymax = bbox.tolist()
            xmin *= scale_x
            xmax *= scale_x
            ymin *= scale_y
            ymax *= scale_y
            bbox = torch.tensor([xmin, xmax, ymin, ymax], dtype=bbox.dtype, device=bbox.device)

        return image, bbox, q_gt, t_gt

transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

syntheticdataset = Speed(images_dir='/kaggle/input/speedsplit/speed/images/trainval', json_dir='/kaggle/input/speedsplit/speed/val.json', transform=transform)
realdataset = Speed(images_dir='/kaggle/input/speedsplit/speed/images/real', json_dir='/kaggle/input/speedsplit/speed/real.json', transform=transform)

class SPN(nn.Module):
    def __init__(self, num_classes=5000, keep_prob=0.5, pretrain=True):
        super().__init__()

        self.num_classes = num_classes
        self.regress_size = self.num_classes
        self.keep_prob = keep_prob

        self.rpn_pre_nms_top_n_train = 500
        self.rpn_pre_nms_top_n_test = 250
        self.rpn_post_nms_top_n_train = 100
        self.rpn_post_nms_top_n_test = 50
        self.anchor_sizes   = ((32, 64,128),)
        self.aspect_ratios  = ((0.5, 1.0,2.0),)
        self.num_anchors = len(self.anchor_sizes[0]) * len(self.aspect_ratios[0])

        self.min_size = 224
        self.max_size = 224
        self.image_mean = [0.485, 0.456, 0.406]
        self.image_std = [0.229, 0.224, 0.225]

        self.transform = GeneralizedRCNNTransform(
            min_size=self.min_size,
            max_size=self.max_size,
            image_mean=self.image_mean,
            image_std=self.image_std
        )

        # 1st Layer: Conv (w ReLu) -> Pool -> Lrn
        self.conv1 = nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0) # Valid padding
        self.pool1 = nn.MaxPool2d(3, stride=2, padding=0)
        self.norm1 = nn.LocalResponseNorm(2, alpha=2e-5, beta=0.75, k=1.0)

        # 2nd Layer: Conv (w ReLu) -> Pool -> Lrn with 2 groups
        self.conv2 = nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2, groups=2)
        self.pool2 = nn.MaxPool2d(3, stride=2, padding=0)
        self.norm2 = nn.LocalResponseNorm(2, alpha=2e-5, beta=0.75, k=1.0)

        # 3rd Layer: Conv (w ReLU)
        self.conv3 = nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1)

        # 4th Layer: Conv (w ReLu) splitted into two groups
        self.conv4 = nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1, groups=2)

        # 5th Layer: Conv (w ReLu) -> Pool splitted into two groups
        self.conv5 = nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1, groups=2)
        self.pool5 = nn.MaxPool2d(3, stride=2, padding=0)

        self.rpn = RegionProposalNetwork(
                anchor_generator=AnchorGenerator(sizes=self.anchor_sizes, aspect_ratios=self.aspect_ratios),
                head=RPNHead(in_channels=256, num_anchors=self.num_anchors),
                fg_iou_thresh=0.7,
                bg_iou_thresh=0.3,
                batch_size_per_image=128,
                positive_fraction=0.5,
                pre_nms_top_n=dict(training=self.rpn_pre_nms_top_n_train, testing=self.rpn_pre_nms_top_n_test),
                post_nms_top_n=dict(training=self.rpn_post_nms_top_n_train, testing=self.rpn_post_nms_top_n_test),
                nms_thresh=0.7,
                score_thresh=0.0,
        )

        self.roi_heads = RoIHeads(
                    box_roi_pool=ops.MultiScaleRoIAlign(featmap_names=['0'], output_size=(6, 6), sampling_ratio=2),
                    box_head=TwoMLPHead(in_channels=256 * 6 * 6, representation_size=1024),
                    box_predictor=FastRCNNPredictor(in_channels=1024, num_classes=2),
                    fg_iou_thresh=0.5,
                    bg_iou_thresh=0.5,
                    batch_size_per_image=64,
                    positive_fraction=0.25,
                    bbox_reg_weights=None,
                    score_thresh=0.05,
                    nms_thresh=0.5,
                    detections_per_img=1,
                )

        self.roi_pool_size = 256 * 6 * 6


        # 6th Layer: Flatten -> FC (w ReLu) -> Dropout
        self.fc6 = nn.Linear(self.roi_pool_size, 4096)
        self.dropout6 = nn.Dropout(p=self.keep_prob, inplace=False)

        # 7th Layer: FC (w ReLu) -> Dropout
        self.fc7 = nn.Linear(4096, 4096)
        self.dropout7 = nn.Dropout(p=self.keep_prob, inplace=False)

        # 8th Layer: FC (no ReLu), return unscaled activations (for tf.nn.softmax_cross_entropy_with_logits)
        self.fc8 = nn.Linear(4096, self.num_classes)

        # 9th Layer: Flatten -> FC (w ReLu) -> Dropout
        self.fc9 = nn.Linear(self.roi_pool_size, 4096)
        self.dropout9 = nn.Dropout(p=self.keep_prob, inplace=False)

        # 10th Layer: FC (w ReLu) -> Dropout
        self.fc10 = nn.Linear(4096, 4096)
        self.dropout10 = nn.Dropout(p=self.keep_prob, inplace=False)

        # 11th Layer: FC (no ReLu), return unscaled activations
        self.fc11 = nn.Linear(4096, self.regress_size)


        if pretrain:
            state_dict = torch.load("/kaggle/input/spnv300classes/epoch_55_model_weights.pth", map_location=device)
            self.load_state_dict(state_dict, strict=False)            

    def forward(self, X, targets=None):

        images, targets = self.transform(X, targets)

        x = self.norm1(self.pool1(F.relu(self.conv1(images.tensors))))
        x = self.norm2(self.pool2(F.relu(self.conv2(x))))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        conv_features = self.pool5(F.relu(self.conv5(x)))

        features = {'0': conv_features}

        #RPN
        proposals, proposal_losses = self.rpn(images, features, targets=targets)
        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets=targets)

        boxes_list = [det['boxes'] for det in detections]

        if len(boxes_list) == 0 or all(b.shape[0] == 0 for b in boxes_list):
            if targets is None:
                default_boxes = []
                for i, img_size in enumerate(images.image_sizes):
                    h, w = img_size
                    default_box = torch.tensor([[0, 0, w, h]], 
                                              device=X.device, 
                                              dtype=torch.float32)
                    default_boxes.append(default_box)
                boxes_list = default_boxes
            else:
                boxes_list = [t['boxes'] for t in targets]

        pooled = self.roi_heads.box_roi_pool(
            features,
            boxes_list,
            images.image_sizes,
        )
        
        pooled_features = pooled.flatten(start_dim=1)


        # Attitude classification
        c = self.dropout6(F.relu(self.fc6(pooled_features)))
        c = self.dropout7(F.relu(self.fc7(c)))
        c = self.fc8(c)

        # Attitude regression
        r = self.dropout9(F.relu(self.fc9(pooled_features)))
        r = self.dropout10(F.relu(self.fc10(r)))
        r = self.fc11(r)

        return c, r, detections
    
def test_loop(test_dataloader, model, device, cameraMatrix, distCoeffs, corners3D, qClass, numNeighbors=25):
    # Initialize error meters and lists for median
    err_q_meter     = AverageMeter('deg')
    err_t_meter     = AverageMeter('m',is_vector=True)
    err_iou_meter = AverageMeter('iou')
    q_errors_all = []
    t_errors_mag_all = []
    iou_errors_all = []

    model.eval()
    
    for batch,(X, yBbox, q_gt, t_gt) in enumerate(test_dataloader):
        X, yBbox, q_gt, t_gt = X.to(device), yBbox.to(device), q_gt.to(device), t_gt.to(device)
        B = X.shape[0]
        model.eval()

        with torch.no_grad():
            classes, weights, detections = model(X)

            topWeights, topClasses = torch.topk(weights, numNeighbors, dim=1)
            topWeights = torch.softmax(topWeights, dim=1)

        
        qs_pr = qClass[topClasses[0].cpu()] 

        # Weighted mean
        q_pr = weighted_mean_quaternion(qs_pr, topWeights.cpu().squeeze())

        # Position
        t_pr = compute_position_spn(q_pr, yBbox[0].numpy(), corners3D, cameraMatrix, distCoeffs)

        # Ground-truth
        q_gt_i = q_gt[0].numpy()
        t_gt_i = t_gt[0].numpy()

        # Metrics
        err_q = error_orientation(q_pr, q_gt_i) # [deg]
        err_t = error_translation(t_pr, t_gt_i)

        # Collect errors for median
        q_errors_all.append(err_q)
        t_errors_mag_all.append(np.linalg.norm(err_t))
         
        # IOU calculation - we know there's only one detection to process
        if detections and len(detections[0]['boxes']) > 0:
            # Get the predicted box
            pred_box = detections[0]['boxes'][0].cpu().numpy()
            gt_box = yBbox[0].cpu().numpy()
            iou = calculate_iou(gt_box, pred_box)
            err_iou_meter.update(iou, 1)
            iou_errors_all.append(iou)

        err_q_meter.update(err_q, B)
        err_t_meter.update(err_t, B)

        print(f"\rBatch {batch+1}/{len(test_dataloader)}: Orientation Error: {err_q_meter.val:.2f} {err_q_meter.unit}, Translation Error: [{err_t[0]:.2f}, {err_t[1]:.2f}, {err_t[2]:.2f}] (mag: {np.linalg.norm(err_t_meter.val):.2f}) {err_t_meter.unit}, IoU: {err_iou_meter.val:.2f}      ", end="", flush=True)


    performances = {
        'eR': err_q_meter,
        'eT': err_t_meter,
        'IoU': err_iou_meter
    }
    # Compute median metrics
    medians = {
        'eR_med': np.median(q_errors_all),
        'eT_med': np.median(t_errors_mag_all),
        'IoU_med': np.median(iou_errors_all) if iou_errors_all else float('nan')
    }
    return performances, medians
       
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=SPN().to(device)

batch_size = 1

test_syn_dataloader = DataLoader(syntheticdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
test_real_dataloader = DataLoader(realdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)

print("Testing on synthetic dataset...")
performances_syn, medians_syn = test_loop(test_syn_dataloader, model, device, syntheticdataset.cameraMatrix, syntheticdataset.distCoeffs, syntheticdataset.keypts3d, syntheticdataset.attClassesMAT)
print("\n")
print("\nTesting on real dataset...")
performances_real, medians_real = test_loop(test_real_dataloader, model, device, realdataset.cameraMatrix, realdataset.distCoeffs, realdataset.keypts3d, realdataset.attClassesMAT)
print("\n")


print("\nResults Summary:")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Metric':<20} | {'SPEED synthetic test-set':<28} | {'SPEED real test-set':<28} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Mean IoU (-)':<20} | {performances_syn['IoU'].avg: <28.4f} | {performances_real['IoU'].avg: <28.4f} |")
print(f"| {'Median IoU (-)':<20} | {medians_syn['IoU_med']: <28.4f} | {medians_real['IoU_med']: <28.4f} |")

mean_et_syn = f"[{performances_syn['eT'].avg[0]:.3f} {performances_syn['eT'].avg[1]:.3f} {performances_syn['eT'].avg[2]:.3f}]"
mean_et_real = f"[{performances_real['eT'].avg[0]:.3f} {performances_real['eT'].avg[1]:.3f} {performances_real['eT'].avg[2]:.3f}]"
print(f"| {'Mean ET (m)':<20} | {mean_et_syn:<28} | {mean_et_real:<28} |")


et_mag_syn = np.linalg.norm(performances_syn['eT'].avg)
et_mag_real = np.linalg.norm(performances_real['eT'].avg)
print(f"| {'Mean ET mag (m)':<20} | {et_mag_syn:<28.4f} | {et_mag_real:<28.4f} |")


median_et_syn = f"{medians_syn['eT_med']:.3f}"
median_et_real = f"{medians_real['eT_med']:.3f}"
print(f"| {'Median ET mag (m)':<20} | {median_et_syn:<28} | {median_et_real:<28} |")


print(f"| {'Mean ER (deg)':<20} | {performances_syn['eR'].avg:<28.4f} | {performances_real['eR'].avg:<28.4f} |")
print(f"| {'Median ER (deg)':<20} | {medians_syn['eR_med']:<28.4f} | {medians_real['eR_med']:<28.4f} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")

100%|██████████| 5/5 [00:00<00:00, 2466.08it/s]


Testing on synthetic dataset...
Batch 2400/2400: Orientation Error: 8.52 deg, Translation Error: [18.38, 10.54, -53.58] (mag: 57.62) m, IoU: 0.89      


Testing on real dataset...
Batch 5/5: Orientation Error: 171.03 deg, Translation Error: [6.38, 3.96, -18.11] (mag: 19.60) m, IoU: 0.71      


Results Summary:
+----------------------+------------------------------+------------------------------+
| Metric               | SPEED synthetic test-set     | SPEED real test-set          |
+----------------------+------------------------------+------------------------------+
| Mean IoU (-)         | 0.7238                       | 0.7042                       |
| Median IoU (-)       | 0.7734                       | 0.7068                       |
| Mean ET (m)          | [19.742 11.363 -57.231]      | [5.899 3.588 -16.906]        |
| Mean ET mag (m)      | 61.5972                      | 18.2612                      |
| Median ET mag (m)    | 53.810                       | 19.603               